# 常暗之厢 — 四步渐进式解析工作流

**新流程**（2026-05-14）：source.txt → `run_pipeline()` → L1+L2+L3 JSON

旧 `parse_module()` 已被四步渐进式流程替代。新流程每步输出中间结果，方便调试和验证。

**流程概览**：
1. Step 1：名称固化 + 精修模组（2 calls 并行）
2. Step 2：内容生成 — interactions 先跑 → events + auto_triggers + L1 + L3 并行
3. Step 3：依赖解析 + 交叉核对（2 calls 串行）
4. Step 4：Library 匹配

**总计**：10 LLM calls / 6 串行步。每步含 `_with_fallback` 保底策略（重试 3 次 → 降级输出）。

In [1]:
import sys
import json
import os

sys.path.insert(0, "../src")

from utils import parser, estimate_and_truncate_context
from module_designer import (
    # Pipeline
    run_pipeline, PipelineResult, save_pipeline_result,
    # Schema
    validate_all, SchemaReport,
    # Data models
    SceneL1, SceneL2, AutoTrigger,
    L3Designer, WorldRule, SceneIntent,
)
from library import WeaponLibrary, EnemyLibrary
from llm import call_deepseek

In [2]:
# 加载模组源文档
content = parser("../常暗之厢（7版规则，简体修正版）.docx")
content = estimate_and_truncate_context(content)

print(f"源文档长度: {len(content)} 字符 (~{len(content)//2} tokens 估算)")

[Token 预估] content: 12,458 tokens
[Token 预估] 合计: 12,458 tokens (上限: 300,000)
[Token 预估] 无需截断，直接使用原文
源文档长度: 9674 字符 (~4837 tokens 估算)


In [3]:
# 初始化武器库和敌人库（供 Step 4 Library 匹配使用）
wl = WeaponLibrary()
wl.load_core()
el = EnemyLibrary()
el.load_core()

print(f"武器库：{len(wl.list_all())} 件")
for w in wl.list_all():
    print(f"  - {w.name}")
print(f"敌人库：{len(el.list_all())} 个")
for e in el.list_all():
    print(f"  - {e.name}")

武器库：10 件
  - 拳头/脚踢
  - 小刀
  - .45自动手枪
  - .38左轮手枪
  - 霰弹枪(12号)
  - 手电筒
  - 消防斧
  - 撬棍
  - 警棍
  - 步枪(.30-06)
敌人库：5 个
  - Clicker
  - 大嘴吞噬者
  - 深潜者
  - 食尸鬼
  - 疯狂信徒


In [ ]:
# Pipeline 需要两个 LLM 适配器：
#   llm_json: (prompt, system) → dict   — 用于 JSON 模式调用
#   llm_text: (prompt, system) → str    — 用于文本模式调用（Step 1b 返回 markdown）

def llm_json(prompt_text: str, system: str = None) -> dict:
    """JSON 模式：返回 dict。用于 Step 1a / 2a / 2b / 2c / 3a / 3b / 4
    call_deepseek(json_mode=True) → temperature=0.3, max_tokens=162840"""
    return call_deepseek(prompt_text, system=system, json_mode=True)

def llm_text(prompt_text: str, system: str = None) -> str:
    """文本模式：返回 str。用于 Step 1b（精修模组输出 markdown）
    call_deepseek(json_mode=False) → temperature=0.7, max_tokens=20000"""
    return call_deepseek(prompt_text, system=system, json_mode=False)

print("LLM 适配器就绪")
print(f"  llm_json: {call_deepseek.__name__}(json_mode=True)")
print(f"  llm_text: {call_deepseek.__name__}(json_mode=False)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 执行四步渐进式解析管线
# ═══════════════════════════════════════════════════════════════
# run_pipeline() 的参数：
#   content         — 原始模组文档
#   llm_json        — JSON 模式 LLM 调用 (prompt, system) → dict
#   llm_text        — 文本模式 LLM 调用 (prompt, system) → str
#   weapon_lib      — 武器库（供 Step 4 使用）
#   enemy_lib       — 敌人库（供 Step 4 使用）
#   max_retries     — 每步最大重试次数（默认 3）
#   verbose         — 是否打印详细进度
#
# 返回 PipelineResult，包含：
#   .step1_data     — Step 1 的完整输出 (meta + scenes + characters + condensed_text)
#   .l1_data        — 最终 L1 数据
#   .l2_data        — 最终 L2 数据 (interactions + events + auto_triggers)
#   .l3_data        — 最终 L3 数据
#   .schema_reports — 各层 Schema 验证报告
#   .cross_ref_report — 交叉引用验证报告
#   .fallbacks      — 触发保底策略的步骤列表

MODULE_NAME = "常暗之厢"
MODULE_DIR = f"../data/modules/{MODULE_NAME}"

result = run_pipeline(
    content,
    llm_json=llm_json,
    llm_text=llm_text,
    weapon_lib=wl,
    enemy_lib=el,
    max_retries=3,
    verbose=True,
)

══════════════════════════════════════════════════
[Step 1] 元信息提取 + 精修模组...
  Step 1 完成: 7 场景, 3 角色
══════════════════════════════════════════════════
[Step 2a] Interactions 提取...


## 中间结果检查：Step 1 — 名称固化 + 精修模组

Step 1 产出了场景 ID 列表和精修模组文本，这是后续所有步骤的基础。

In [ ]:
# Step 1 中间结果
step1 = result.step1_data
print("─── 场景列表 ───")
for s in step1.get("scenes", []):
    print(f"  {s['id']}: {s['name']}")
print(f"\n─── 角色列表 ───")
for c in step1.get("characters", []):
    print(f"  {c['id']}: {c['name']}")
print(f"\n─── 精修模组 (前 800 字) ───")
ct = step1.get("condensed_text", "")
print(ct[:800] + ("..." if len(ct) > 800 else ""))
print(f"\n总长度: {len(ct)} 字符")

# 检查是否有保底触发
if any("Step 1" in fb for fb in result.fallbacks):
    print("\n⚠ Step 1 触发了保底策略")

## 中间结果检查：Step 2 — 内容生成

查看 interactions、events、auto_triggers、L1、L3 的产出概况。

In [ ]:
# 按场景统计 interactions
from collections import Counter
inter_by_scene = Counter(i.get("scene", "?") for i in result.l2_data.get("interactions", []))
print(f"─── Interactions: {len(result.l2_data['interactions'])} 个 ───")
for sid, count in sorted(inter_by_scene.items()):
    print(f"  {sid}: {count} 个互动")
    for i in result.l2_data["interactions"]:
        if i.get("scene") == sid:
            flag_info = [s.get("key","") for s in i.get("side_effects", []) if s.get("type") == "flag_set"]
            print(f"    {i['id']}: {i['name']}" + (f" (flag: {flag_info})" if flag_info else ""))

print(f"\n─── Events: {len(result.l2_data.get('events', []))} 个 ───")
for ev in result.l2_data.get("events", []):
    print(f"  {ev.get('id', '?')}: {ev.get('name', '?')}")

print(f"\n─── Auto-triggers: {len(result.l2_data.get('auto_triggers', []))} 个 ───")
for at in result.l2_data.get("auto_triggers", []):
    print(f"  {at.get('id', '?')}: {at.get('name', '?')} → {at.get('effect_type', '?')}")

print(f"\n─── L1: {len(result.l1_data)} 个场景 ───")
for name, sdata in result.l1_data.items():
    print(f"  {name}: {len(sdata.get('perceptible', []))} 感知元素, {len(sdata.get('npc_appearances', []))} NPC")

print(f"\n─── L3: {len(result.l3_data.get('world_rules', []))} 世界规则, {len(result.l3_data.get('scene_intents', {}))} 场景意图 ───")

## Schema 验证 + 交叉引用报告

In [ ]:
# Schema 验证结果
print("═══ Schema 验证 ═══")
for layer, report in result.schema_reports.items():
    status = "PASS" if report.is_valid else "ISSUES"
    print(f"  {layer} [{status}]: {report.summary()}")

# 交叉引用报告
print("\n═══ 交叉引用 ═══")
if result.cross_ref_report:
    print(f"  {result.cross_ref_report.summary()}")
else:
    print("  未执行交叉引用验证")

# 保底策略
if result.fallbacks:
    print(f"\n═══ 保底触发 ═══")
    for fb in result.fallbacks:
        print(f"  ⚠ {fb}")

In [ ]:
# 保存管线结果到模块目录
save_pipeline_result(result, MODULE_DIR)
print(f"\n管线结果已保存至 {MODULE_DIR}/")

## 最终统计

生成结果总览

In [ ]:
print("=" * 60)
print("四步渐进式解析完成")
print("=" * 60)
print(f"Step 1: {len(result.step1_data.get('scenes', []))} 场景, {len(result.step1_data.get('characters', []))} 角色")
print(f"         condensed_text {len(result.step1_data.get('condensed_text', ''))} 字符")
print(f"Step 2: {len(result.l2_data.get('interactions', []))} interactions")
print(f"        {len(result.l2_data.get('events', []))} events")
print(f"        {len(result.l2_data.get('auto_triggers', []))} auto_triggers")
print(f"        {len(result.l1_data)} L1 场景")
print(f"        {len(result.l3_data.get('world_rules', []))} 世界规则")
print(f"Step 3-4: Library 匹配已{'完成' if wl or el else '跳过'}")
print(f"")
print(f"总 LLM 调用: 10 (Step 1:2 + Step 2:5 + Step 3:2 + Step 4:1)")
print(f"保底触发: {len(result.fallbacks)} 处")
print(f"状态: {'PASS' if result.all_valid else 'HAS_ISSUES'}")